In [ ]:
# =============================================================================
# Paper 1 — Modern Baseline 1
# Vanilla Masked Autoencoder (MAE), 13x13 primary configuration
#
# CELL 1
# Environment, frozen paths, reproducibility, and input-data audit
#
# IMPORTANT
# ---------
# This notebook MUST NOT:
#   - regenerate patches
#   - refit scalers
#   - modify spatial folds
#   - use geology, GPI, or warm-spring information
#   - overwrite PB-MMAE outputs
#
# It uses the already frozen fold-specific 13x13 tensors.
# =============================================================================

from pathlib import Path
import random
import platform

import numpy as np
import pandas as pd
import torch


# -----------------------------------------------------------------------------
# 1. Project configuration
# -----------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\project 3\Paper1_SSL")

PATCH_ROOT = (
    PROJECT_ROOT
    / "03_outputs"
    / "patches"
    / "13x13"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "03_outputs"
    / "modern_baselines"
    / "MAE_13x13"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

FOLDS = [1, 2, 3, 4, 5]

PATCH_SIZE = 13
N_CHANNELS = 14

# Must remain identical to PB-MMAE primary spatial masking:
N_TOKENS = PATCH_SIZE * PATCH_SIZE          # 169
N_MASKED = 68
MASK_RATIO = N_MASKED / N_TOKENS           # 0.402366...

# Frozen comparison latent dimension
LATENT_DIM = 64

# Development / outer-evaluation roles
DEVELOPMENT_FOLD = 3
OUTER_FOLDS = [1, 2, 4, 5]

# Reproducibility
GLOBAL_SEED = 2026


# -----------------------------------------------------------------------------
# 2. Deterministic seeds
# -----------------------------------------------------------------------------

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GLOBAL_SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# -----------------------------------------------------------------------------
# 3. Environment report
# -----------------------------------------------------------------------------

print("=" * 88)
print("MAE BASELINE — 13x13 FROZEN INPUT AUDIT")
print("=" * 88)

print(f"Project root      : {PROJECT_ROOT}")
print(f"Patch root        : {PATCH_ROOT}")
print(f"Output root       : {OUTPUT_ROOT}")
print(f"Python            : {platform.python_version()}")
print(f"PyTorch           : {torch.__version__}")
print(f"Device            : {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU               : {torch.cuda.get_device_name(0)}")

print()
print(f"Patch size        : {PATCH_SIZE} x {PATCH_SIZE}")
print(f"Channels          : {N_CHANNELS}")
print(f"Spatial tokens    : {N_TOKENS}")
print(f"Masked tokens     : {N_MASKED}")
print(f"Mask ratio        : {MASK_RATIO:.6f} ({MASK_RATIO*100:.2f}%)")
print(f"Latent dimension  : {LATENT_DIM}")
print(f"Development fold  : {DEVELOPMENT_FOLD}")
print(f"Outer folds       : {OUTER_FOLDS}")


# -----------------------------------------------------------------------------
# 4. Audit every frozen fold
# -----------------------------------------------------------------------------

audit_rows = []

for fold in FOLDS:

    fold_dir = PATCH_ROOT / f"Fold_{fold}"

    train_file = (
        fold_dir
        / f"Fold{fold}_TRAIN_13x13_14ch.npy"
    )

    val_file = (
        fold_dir
        / f"Fold{fold}_VALIDATION_13x13_14ch.npy"
    )

    train_meta_file = (
        fold_dir
        / f"Fold{fold}_TRAIN_13x13_metadata.csv"
    )

    val_meta_file = (
        fold_dir
        / f"Fold{fold}_VALIDATION_13x13_metadata.csv"
    )

    required = [
        train_file,
        val_file,
        train_meta_file,
        val_meta_file,
    ]

    missing = [
        str(p)
        for p in required
        if not p.exists()
    ]

    if missing:
        raise FileNotFoundError(
            "Missing frozen input files:\n"
            + "\n".join(missing)
        )

    # Load without changing anything
    X_train = np.load(
        train_file,
        mmap_mode="r"
    )

    X_val = np.load(
        val_file,
        mmap_mode="r"
    )

    train_meta = pd.read_csv(train_meta_file)
    val_meta = pd.read_csv(val_meta_file)

    # -------------------------------------------------------------------------
    # Shape checks
    # -------------------------------------------------------------------------

    expected_train_shape = (
        len(train_meta),
        PATCH_SIZE,
        PATCH_SIZE,
        N_CHANNELS,
    )

    expected_val_shape = (
        len(val_meta),
        PATCH_SIZE,
        PATCH_SIZE,
        N_CHANNELS,
    )

    assert X_train.shape == expected_train_shape, (
        f"Fold {fold} TRAIN shape mismatch:\n"
        f"Tensor   : {X_train.shape}\n"
        f"Expected : {expected_train_shape}"
    )

    assert X_val.shape == expected_val_shape, (
        f"Fold {fold} VALIDATION shape mismatch:\n"
        f"Tensor   : {X_val.shape}\n"
        f"Expected : {expected_val_shape}"
    )

    # -------------------------------------------------------------------------
    # Metadata checks
    # -------------------------------------------------------------------------

    required_meta_columns = {
        "Cell_ID",
        "Row",
        "Col",
        "X",
        "Y",
        "Spatial_Fold",
        "Validation_Fold",
        "CV_Role",
        "Patch_size",
    }

    assert required_meta_columns.issubset(train_meta.columns)
    assert required_meta_columns.issubset(val_meta.columns)

    assert (train_meta["Patch_size"] == 13).all()
    assert (val_meta["Patch_size"] == 13).all()

    assert (train_meta["CV_Role"] == "TRAIN").all()
    assert (val_meta["CV_Role"] == "VALIDATION").all()

    assert (train_meta["Validation_Fold"] == fold).all()
    assert (val_meta["Validation_Fold"] == fold).all()

    # Patch-center Cell_IDs must not overlap
    train_ids = set(train_meta["Cell_ID"].astype(str))
    val_ids = set(val_meta["Cell_ID"].astype(str))

    center_overlap = train_ids.intersection(val_ids)

    assert len(center_overlap) == 0, (
        f"Fold {fold}: TRAIN/VALIDATION center overlap detected."
    )

    # -------------------------------------------------------------------------
    # Numeric checks
    # -------------------------------------------------------------------------

    train_finite = np.isfinite(X_train).all()
    val_finite = np.isfinite(X_val).all()

    assert train_finite, (
        f"Fold {fold}: non-finite values found in TRAIN tensor."
    )

    assert val_finite, (
        f"Fold {fold}: non-finite values found in VALIDATION tensor."
    )

    audit_rows.append(
        {
            "Fold": fold,
            "Role": (
                "DEVELOPMENT"
                if fold == DEVELOPMENT_FOLD
                else "OUTER"
            ),
            "Train_n": X_train.shape[0],
            "Validation_n": X_val.shape[0],
            "Patch_shape": str(X_train.shape[1:]),
            "dtype": str(X_train.dtype),
            "Center_overlap": len(center_overlap),
            "Train_finite": train_finite,
            "Validation_finite": val_finite,
        }
    )


# -----------------------------------------------------------------------------
# 5. Final audit table
# -----------------------------------------------------------------------------

audit_df = pd.DataFrame(audit_rows)

print()
print("=" * 88)
print("FROZEN 13x13 DATA INVENTORY")
print("=" * 88)

display(audit_df)

audit_file = OUTPUT_ROOT / "MAE_13x13_input_audit.csv"

audit_df.to_csv(
    audit_file,
    index=False
)

print()
print(f"Audit saved to:\n{audit_file}")

print()
print("=" * 88)
print("STATUS: INPUT AUDIT PASSED")
print("=" * 88)
print(
    "No patches, scalers, folds, labels, geology, GPI, "
    "or warm-spring data were created or modified."
)

In [ ]:
# =============================================================================
# CELL 2
# Vanilla MAE architecture for 13x13 x 14 multiphysics patches
#
# PURPOSE
# -------
# Modern representation-learning baseline for PB-MMAE.
#
# DESIGN
# ------
# Input              : 13 x 13 x 14
# Spatial tokens     : 169
# Token features     : 14 variables jointly
# Spatial masking    : exactly 68 / 169 tokens = 40.24%
# Whole-family mask  : NONE
# Physics families   : NOT used
# Encoder dimension  : 64
# Encoder blocks     : 4 Transformer blocks
# Attention heads    : 4
# Decoder blocks     : 2
# Output             : reconstruction of the 14 variables at each spatial token
# Benchmark latent   : 64-D mean-pooled FULL unmasked encoder representation
#
# IMPORTANT
# ---------
# This is deliberately a generic masked autoencoder.
# It does NOT use:
#   - family-specific branches
#   - whole-modality masking
#   - physics-balanced loss
#   - geology
#   - GPI
#   - warm springs
# =============================================================================

import math
import numpy as np
import torch
import torch.nn as nn


# -----------------------------------------------------------------------------
# 1. Fixed architecture configuration
# -----------------------------------------------------------------------------

MAE_EMBED_DIM = 64

MAE_ENCODER_DEPTH = 4
MAE_ENCODER_HEADS = 4
MAE_ENCODER_MLP_DIM = 128

MAE_DECODER_DIM = 64
MAE_DECODER_DEPTH = 2
MAE_DECODER_HEADS = 4
MAE_DECODER_MLP_DIM = 128

MAE_DROPOUT = 0.0


# -----------------------------------------------------------------------------
# 2. Fixed 2-D sinusoidal positional encoding
# -----------------------------------------------------------------------------

def _sincos_1d(embed_dim, positions):
    """
    Standard 1-D sine/cosine positional encoding.

    Parameters
    ----------
    embed_dim : int
        Must be even.
    positions : np.ndarray
        Flattened coordinate positions.

    Returns
    -------
    np.ndarray
        Shape: (n_positions, embed_dim)
    """

    assert embed_dim % 2 == 0

    positions = positions.reshape(-1).astype(np.float32)

    omega = np.arange(
        embed_dim // 2,
        dtype=np.float32
    )

    omega /= (embed_dim / 2.0)

    omega = 1.0 / (
        10000.0 ** omega
    )

    angles = np.einsum(
        "m,d->md",
        positions,
        omega
    )

    embedding = np.concatenate(
        [
            np.sin(angles),
            np.cos(angles),
        ],
        axis=1
    )

    return embedding.astype(np.float32)


def make_2d_sincos_position_embedding(
    grid_size,
    embed_dim
):
    """
    Create fixed 2-D positional encoding for a square spatial grid.

    Returns
    -------
    torch.Tensor
        Shape: (1, grid_size * grid_size, embed_dim)
    """

    assert embed_dim % 4 == 0

    rows, cols = np.meshgrid(
        np.arange(grid_size),
        np.arange(grid_size),
        indexing="ij"
    )

    row_embed = _sincos_1d(
        embed_dim // 2,
        rows.reshape(-1)
    )

    col_embed = _sincos_1d(
        embed_dim // 2,
        cols.reshape(-1)
    )

    pos_embed = np.concatenate(
        [
            row_embed,
            col_embed,
        ],
        axis=1
    )

    pos_embed = torch.tensor(
        pos_embed,
        dtype=torch.float32
    ).unsqueeze(0)

    return pos_embed


# -----------------------------------------------------------------------------
# 3. Vanilla MAE
# -----------------------------------------------------------------------------

class VanillaMAE13x13(nn.Module):

    def __init__(
        self,
        patch_size=13,
        n_channels=14,
        embed_dim=64,
        encoder_depth=4,
        encoder_heads=4,
        encoder_mlp_dim=128,
        decoder_dim=64,
        decoder_depth=2,
        decoder_heads=4,
        decoder_mlp_dim=128,
        n_masked=68,
        dropout=0.0,
    ):
        super().__init__()

        self.patch_size = patch_size
        self.n_channels = n_channels

        self.n_tokens = (
            patch_size * patch_size
        )

        self.n_masked = n_masked

        self.n_visible = (
            self.n_tokens - self.n_masked
        )

        self.embed_dim = embed_dim

        # ---------------------------------------------------------------------
        # Input token projection
        #
        # Each 4-km spatial cell is one token.
        # All 14 geophysical variables are presented jointly.
        # ---------------------------------------------------------------------

        self.token_projection = nn.Linear(
            n_channels,
            embed_dim
        )

        # ---------------------------------------------------------------------
        # Fixed 2-D positional encoding
        # ---------------------------------------------------------------------

        pos_embed = (
            make_2d_sincos_position_embedding(
                patch_size,
                embed_dim
            )
        )

        self.register_buffer(
            "encoder_pos_embed",
            pos_embed,
            persistent=True
        )

        # ---------------------------------------------------------------------
        # Transformer encoder
        # ---------------------------------------------------------------------

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=encoder_heads,
            dim_feedforward=encoder_mlp_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=encoder_depth
        )

        self.encoder_norm = nn.LayerNorm(
            embed_dim
        )

        # ---------------------------------------------------------------------
        # Decoder projection
        # ---------------------------------------------------------------------

        self.decoder_projection = nn.Linear(
            embed_dim,
            decoder_dim
        )

        # Learned representation for missing spatial tokens
        self.mask_token = nn.Parameter(
            torch.zeros(
                1,
                1,
                decoder_dim
            )
        )

        decoder_pos_embed = (
            make_2d_sincos_position_embedding(
                patch_size,
                decoder_dim
            )
        )

        self.register_buffer(
            "decoder_pos_embed",
            decoder_pos_embed,
            persistent=True
        )

        # ---------------------------------------------------------------------
        # Lightweight Transformer decoder
        # ---------------------------------------------------------------------

        decoder_layer = nn.TransformerEncoderLayer(
            d_model=decoder_dim,
            nhead=decoder_heads,
            dim_feedforward=decoder_mlp_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.decoder = nn.TransformerEncoder(
            decoder_layer,
            num_layers=decoder_depth
        )

        self.decoder_norm = nn.LayerNorm(
            decoder_dim
        )

        # Reconstruct all 14 variables at each spatial token
        self.reconstruction_head = nn.Linear(
            decoder_dim,
            n_channels
        )

        self._initialize_weights()


    # -------------------------------------------------------------------------
    # Weight initialization
    # -------------------------------------------------------------------------

    def _initialize_weights(self):

        nn.init.normal_(
            self.mask_token,
            std=0.02
        )

        for module in self.modules():

            if isinstance(module, nn.Linear):

                nn.init.xavier_uniform_(
                    module.weight
                )

                if module.bias is not None:
                    nn.init.zeros_(
                        module.bias
                    )


    # -------------------------------------------------------------------------
    # Convert raster patches to spatial token sequence
    # -------------------------------------------------------------------------

    def patch_to_tokens(self, x):

        # Expected:
        # B x 13 x 13 x 14

        if x.ndim != 4:
            raise ValueError(
                f"Expected 4-D input; got {x.shape}"
            )

        B, H, W, C = x.shape

        if (
            H != self.patch_size
            or W != self.patch_size
            or C != self.n_channels
        ):
            raise ValueError(
                "Unexpected patch shape: "
                f"{x.shape}"
            )

        return x.reshape(
            B,
            self.n_tokens,
            self.n_channels
        )


    # -------------------------------------------------------------------------
    # Generate exactly 68 random masked spatial positions per sample
    # -------------------------------------------------------------------------

    def random_mask(self, batch_size, device):

        noise = torch.rand(
            batch_size,
            self.n_tokens,
            device=device
        )

        ids_shuffle = torch.argsort(
            noise,
            dim=1
        )

        ids_keep = ids_shuffle[
            :,
            :self.n_visible
        ]

        mask = torch.ones(
            batch_size,
            self.n_tokens,
            device=device,
            dtype=torch.float32
        )

        mask.scatter_(
            1,
            ids_keep,
            0.0
        )

        return ids_keep, mask


    # -------------------------------------------------------------------------
    # Full unmasked encoder
    #
    # IMPORTANT:
    # This is used when extracting representations for clustering.
    # Every sample therefore receives the same complete 13x13 evidence.
    # -------------------------------------------------------------------------

    def encode_full(self, x):

        tokens = self.patch_to_tokens(x)

        z = self.token_projection(
            tokens
        )

        z = (
            z
            + self.encoder_pos_embed.to(
                dtype=z.dtype
            )
        )

        z = self.encoder(z)

        z = self.encoder_norm(z)

        # One 64-D representation per patch
        latent = z.mean(dim=1)

        return latent


    # -------------------------------------------------------------------------
    # Masked training forward pass
    # -------------------------------------------------------------------------

    def forward(self, x):

        tokens = self.patch_to_tokens(x)

        B = tokens.shape[0]

        target = tokens

        # ---------------------------------------------------------------------
        # Token embedding + spatial position
        # ---------------------------------------------------------------------

        embedded = self.token_projection(
            tokens
        )

        embedded = (
            embedded
            + self.encoder_pos_embed.to(
                dtype=embedded.dtype
            )
        )

        # ---------------------------------------------------------------------
        # Randomly retain exactly 101 of 169 positions
        # ---------------------------------------------------------------------

        ids_keep, mask = self.random_mask(
            B,
            x.device
        )

        gather_index = (
            ids_keep
            .unsqueeze(-1)
            .expand(
                -1,
                -1,
                self.embed_dim
            )
        )

        visible = torch.gather(
            embedded,
            dim=1,
            index=gather_index
        )

        # ---------------------------------------------------------------------
        # Encode visible spatial tokens
        # ---------------------------------------------------------------------

        encoded = self.encoder(
            visible
        )

        encoded = self.encoder_norm(
            encoded
        )

        # Training-time visible-token representation
        visible_latent = encoded.mean(
            dim=1
        )

        # ---------------------------------------------------------------------
        # Decoder input
        #
        # Start with mask tokens everywhere, then put encoded visible
        # representations back into their original spatial positions.
        # ---------------------------------------------------------------------

        decoded_visible = (
            self.decoder_projection(
                encoded
            )
        )

        decoder_tokens = (
            self.mask_token
            .expand(
                B,
                self.n_tokens,
                MAE_DECODER_DIM
            )
            .clone()
        )

        decoder_index = (
            ids_keep
            .unsqueeze(-1)
            .expand(
                -1,
                -1,
                MAE_DECODER_DIM
            )
        )

        decoder_tokens.scatter_(
            dim=1,
            index=decoder_index,
            src=decoded_visible
        )

        decoder_tokens = (
            decoder_tokens
            + self.decoder_pos_embed.to(
                dtype=decoder_tokens.dtype
            )
        )

        decoded = self.decoder(
            decoder_tokens
        )

        decoded = self.decoder_norm(
            decoded
        )

        prediction = (
            self.reconstruction_head(
                decoded
            )
        )

        # ---------------------------------------------------------------------
        # STANDARD MAE LOSS
        #
        # Reconstruction loss is computed ONLY over masked spatial positions.
        #
        # This deliberately distinguishes vanilla MAE from PB-MMAE:
        # vanilla MAE = generic masked-token prediction
        # PB-MMAE     = complete family reconstruction + family balancing
        # ---------------------------------------------------------------------

        token_mse = (
            (prediction - target) ** 2
        ).mean(dim=-1)

        loss = (
            token_mse * mask
        ).sum() / mask.sum()

        return {
            "loss": loss,
            "prediction": prediction,
            "mask": mask,
            "ids_keep": ids_keep,
            "visible_latent": visible_latent,
        }


# -----------------------------------------------------------------------------
# 4. Instantiate model
# -----------------------------------------------------------------------------

mae_model = VanillaMAE13x13(
    patch_size=PATCH_SIZE,
    n_channels=N_CHANNELS,
    embed_dim=MAE_EMBED_DIM,
    encoder_depth=MAE_ENCODER_DEPTH,
    encoder_heads=MAE_ENCODER_HEADS,
    encoder_mlp_dim=MAE_ENCODER_MLP_DIM,
    decoder_dim=MAE_DECODER_DIM,
    decoder_depth=MAE_DECODER_DEPTH,
    decoder_heads=MAE_DECODER_HEADS,
    decoder_mlp_dim=MAE_DECODER_MLP_DIM,
    n_masked=N_MASKED,
    dropout=MAE_DROPOUT,
).to(DEVICE)


# -----------------------------------------------------------------------------
# 5. Parameter audit
# -----------------------------------------------------------------------------

trainable_params = sum(
    p.numel()
    for p in mae_model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in mae_model.parameters()
)

print("=" * 88)
print("VANILLA MAE — ARCHITECTURE AUDIT")
print("=" * 88)

print(mae_model)

print()
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")


# -----------------------------------------------------------------------------
# 6. Smoke test
# -----------------------------------------------------------------------------

dummy = torch.randn(
    2,
    PATCH_SIZE,
    PATCH_SIZE,
    N_CHANNELS,
    device=DEVICE
)

mae_model.eval()

with torch.no_grad():

    output = mae_model(dummy)

    full_latent = mae_model.encode_full(
        dummy
    )


print()
print("=" * 88)
print("FORWARD-PASS AUDIT")
print("=" * 88)

print(
    "Prediction shape    :",
    tuple(output["prediction"].shape)
)

print(
    "Mask shape          :",
    tuple(output["mask"].shape)
)

print(
    "Masked/token counts :",
    output["mask"]
    .sum(dim=1)
    .int()
    .cpu()
    .tolist()
)

print(
    "Visible latent      :",
    tuple(
        output["visible_latent"].shape
    )
)

print(
    "Full latent         :",
    tuple(full_latent.shape)
)

print(
    "Smoke-test loss     :",
    float(output["loss"].cpu())
)


# -----------------------------------------------------------------------------
# 7. Hard assertions
# -----------------------------------------------------------------------------

assert output["prediction"].shape == (
    2,
    169,
    14
)

assert output["mask"].shape == (
    2,
    169
)

assert torch.all(
    output["mask"].sum(dim=1)
    == N_MASKED
)

assert output["visible_latent"].shape == (
    2,
    64
)

assert full_latent.shape == (
    2,
    64
)

assert torch.isfinite(
    output["loss"]
)

assert torch.isfinite(
    full_latent
).all()


print()
print("=" * 88)
print("STATUS: MAE ARCHITECTURE AUDIT PASSED")
print("=" * 88)

In [ ]:
# =============================================================================
# CELL 3
# Train Vanilla MAE on Fold 3 DEVELOPMENT partition only
#
# IMPORTANT
# ---------
# - Uses frozen Fold 3 tensors from Cell 1.
# - Does NOT regenerate patches or scaling.
# - Does NOT use geology, GPI, warm springs, or cluster labels.
# - Fixed 300-epoch training budget.
# - No early stopping.
# - Validation loss is diagnostic only.
# =============================================================================

from pathlib import Path
import copy
import csv
import random
import time

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader


# -----------------------------------------------------------------------------
# 1. Frozen training configuration
# -----------------------------------------------------------------------------

TRAIN_FOLD = 3

MAE_EPOCHS = 300
MAE_BATCH_SIZE = 32
MAE_LR = 1e-3
MAE_WEIGHT_DECAY = 1e-4

MAE_NUM_WORKERS = 0   # safest for Windows/Jupyter
MAE_PIN_MEMORY = torch.cuda.is_available()

TRAIN_SEED = 2026


# -----------------------------------------------------------------------------
# 2. Reproducibility reset
# -----------------------------------------------------------------------------

random.seed(TRAIN_SEED)
np.random.seed(TRAIN_SEED)
torch.manual_seed(TRAIN_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(TRAIN_SEED)

# Deterministic behavior where supported
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# -----------------------------------------------------------------------------
# 3. Fold 3 paths
# -----------------------------------------------------------------------------

fold_dir = (
    PATCH_ROOT
    / f"Fold_{TRAIN_FOLD}"
)

train_file = (
    fold_dir
    / f"Fold{TRAIN_FOLD}_TRAIN_13x13_14ch.npy"
)

val_file = (
    fold_dir
    / f"Fold{TRAIN_FOLD}_VALIDATION_13x13_14ch.npy"
)

train_meta_file = (
    fold_dir
    / f"Fold{TRAIN_FOLD}_TRAIN_13x13_metadata.csv"
)

val_meta_file = (
    fold_dir
    / f"Fold{TRAIN_FOLD}_VALIDATION_13x13_metadata.csv"
)

fold_output_dir = (
    OUTPUT_ROOT
    / "Fold_3_DEVELOPMENT"
)

fold_output_dir.mkdir(
    parents=True,
    exist_ok=True
)


# -----------------------------------------------------------------------------
# 4. Dataset
# -----------------------------------------------------------------------------

class PatchDataset(Dataset):

    def __init__(self, npy_path):

        self.x = np.load(
            npy_path,
            mmap_mode="r"
        )

    def __len__(self):

        return self.x.shape[0]

    def __getitem__(self, idx):

        # Make a writable contiguous copy to avoid NumPy memmap warnings
        sample = np.array(
            self.x[idx],
            dtype=np.float32,
            copy=True
        )

        return torch.from_numpy(sample)


train_dataset = PatchDataset(
    train_file
)

val_dataset = PatchDataset(
    val_file
)


# -----------------------------------------------------------------------------
# 5. DataLoaders
# -----------------------------------------------------------------------------

loader_generator = torch.Generator()
loader_generator.manual_seed(TRAIN_SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=MAE_BATCH_SIZE,
    shuffle=True,
    num_workers=MAE_NUM_WORKERS,
    pin_memory=MAE_PIN_MEMORY,
    generator=loader_generator,
    drop_last=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=MAE_BATCH_SIZE,
    shuffle=False,
    num_workers=MAE_NUM_WORKERS,
    pin_memory=MAE_PIN_MEMORY,
    drop_last=False
)


print("=" * 88)
print("FOLD 3 MAE TRAINING CONFIGURATION")
print("=" * 88)

print(f"TRAIN samples      : {len(train_dataset)}")
print(f"VALIDATION samples : {len(val_dataset)}")
print(f"Batch size         : {MAE_BATCH_SIZE}")
print(f"Epochs             : {MAE_EPOCHS}")
print(f"Learning rate      : {MAE_LR}")
print(f"Weight decay       : {MAE_WEIGHT_DECAY}")
print(f"Device             : {DEVICE}")
print(f"Output directory   : {fold_output_dir}")


# -----------------------------------------------------------------------------
# 6. Re-initialize a fresh MAE
#
# Do not continue from the Cell 2 smoke-test model.
# -----------------------------------------------------------------------------

random.seed(TRAIN_SEED)
np.random.seed(TRAIN_SEED)
torch.manual_seed(TRAIN_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(TRAIN_SEED)


model = VanillaMAE13x13(
    patch_size=PATCH_SIZE,
    n_channels=N_CHANNELS,
    embed_dim=MAE_EMBED_DIM,
    encoder_depth=MAE_ENCODER_DEPTH,
    encoder_heads=MAE_ENCODER_HEADS,
    encoder_mlp_dim=MAE_ENCODER_MLP_DIM,
    decoder_dim=MAE_DECODER_DIM,
    decoder_depth=MAE_DECODER_DEPTH,
    decoder_heads=MAE_DECODER_HEADS,
    decoder_mlp_dim=MAE_DECODER_MLP_DIM,
    n_masked=N_MASKED,
    dropout=MAE_DROPOUT,
).to(DEVICE)


# -----------------------------------------------------------------------------
# 7. Optimizer
# -----------------------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=MAE_LR,
    weight_decay=MAE_WEIGHT_DECAY
)


# -----------------------------------------------------------------------------
# 8. Validation function
#
# Validation is diagnostic only.
# A fixed RNG state is used each epoch so that the SAME sequence of validation
# masks is evaluated every epoch, reducing mask-induced noise.
# -----------------------------------------------------------------------------

@torch.no_grad()
def evaluate_masked_loss(
    model,
    loader,
    seed=123456
):

    model.eval()

    # Preserve current RNG state
    cpu_rng_state = torch.get_rng_state()

    cuda_rng_state = None

    if torch.cuda.is_available():

        cuda_rng_state = (
            torch.cuda.get_rng_state_all()
        )

    # Fixed validation masking sequence
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    total_loss = 0.0
    total_n = 0

    for batch in loader:

        batch = batch.to(
            DEVICE,
            non_blocking=True
        )

        output = model(batch)

        batch_n = batch.shape[0]

        total_loss += (
            float(output["loss"].item())
            * batch_n
        )

        total_n += batch_n

    # Restore RNG state
    torch.set_rng_state(
        cpu_rng_state
    )

    if (
        torch.cuda.is_available()
        and cuda_rng_state is not None
    ):
        torch.cuda.set_rng_state_all(
            cuda_rng_state
        )

    return total_loss / total_n


# -----------------------------------------------------------------------------
# 9. Training loop
# -----------------------------------------------------------------------------

history = []

training_start = time.time()

for epoch in range(
    1,
    MAE_EPOCHS + 1
):

    epoch_start = time.time()

    model.train()

    train_loss_sum = 0.0
    train_n = 0

    for batch in train_loader:

        batch = batch.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        output = model(batch)

        loss = output["loss"]

        if not torch.isfinite(loss):

            raise RuntimeError(
                f"Non-finite loss at epoch {epoch}"
            )

        loss.backward()

        optimizer.step()

        batch_n = batch.shape[0]

        train_loss_sum += (
            float(loss.item())
            * batch_n
        )

        train_n += batch_n


    train_loss = (
        train_loss_sum / train_n
    )


    # -------------------------------------------------------------------------
    # Diagnostic validation
    # -------------------------------------------------------------------------

    val_loss = evaluate_masked_loss(
        model,
        val_loader,
        seed=123456
    )


    epoch_seconds = (
        time.time()
        - epoch_start
    )


    history.append(
        {
            "epoch": epoch,
            "train_masked_mse": train_loss,
            "validation_masked_mse": val_loss,
            "epoch_seconds": epoch_seconds,
        }
    )


    # -------------------------------------------------------------------------
    # Compact logging
    # -------------------------------------------------------------------------

    if (
        epoch == 1
        or epoch % 10 == 0
        or epoch == MAE_EPOCHS
    ):

        print(
            f"Epoch {epoch:03d}/{MAE_EPOCHS} | "
            f"train={train_loss:.8f} | "
            f"val={val_loss:.8f} | "
            f"{epoch_seconds:.2f}s"
        )


# -----------------------------------------------------------------------------
# 10. Save final model
#
# IMPORTANT:
# Final epoch is retained because the training budget is fixed.
# Validation loss is NOT used for checkpoint selection.
# -----------------------------------------------------------------------------

training_seconds = (
    time.time()
    - training_start
)

checkpoint_path = (
    fold_output_dir
    / "MAE_Fold3_13x13_final_epoch300.pt"
)

torch.save(
    {
        "model_state_dict":
            model.state_dict(),

        "architecture": {
            "patch_size": PATCH_SIZE,
            "n_channels": N_CHANNELS,
            "n_tokens": N_TOKENS,
            "n_masked": N_MASKED,
            "mask_ratio": MASK_RATIO,
            "embed_dim": MAE_EMBED_DIM,
            "encoder_depth": MAE_ENCODER_DEPTH,
            "encoder_heads": MAE_ENCODER_HEADS,
            "encoder_mlp_dim": MAE_ENCODER_MLP_DIM,
            "decoder_dim": MAE_DECODER_DIM,
            "decoder_depth": MAE_DECODER_DEPTH,
            "decoder_heads": MAE_DECODER_HEADS,
            "decoder_mlp_dim": MAE_DECODER_MLP_DIM,
        },

        "training": {
            "fold": TRAIN_FOLD,
            "epochs": MAE_EPOCHS,
            "batch_size": MAE_BATCH_SIZE,
            "learning_rate": MAE_LR,
            "weight_decay": MAE_WEIGHT_DECAY,
            "seed": TRAIN_SEED,
            "training_seconds": training_seconds,
        },
    },
    checkpoint_path
)


# -----------------------------------------------------------------------------
# 11. Save training history
# -----------------------------------------------------------------------------

history_df = pd.DataFrame(
    history
)

history_path = (
    fold_output_dir
    / "MAE_Fold3_13x13_training_history.csv"
)

history_df.to_csv(
    history_path,
    index=False
)


# -----------------------------------------------------------------------------
# 12. Final numerical checks
# -----------------------------------------------------------------------------

assert len(history_df) == MAE_EPOCHS

assert np.isfinite(
    history_df["train_masked_mse"]
).all()

assert np.isfinite(
    history_df[
        "validation_masked_mse"
    ]
).all()

assert checkpoint_path.exists()

assert history_path.exists()


# -----------------------------------------------------------------------------
# 13. Summary
# -----------------------------------------------------------------------------

best_val_row = (
    history_df.loc[
        history_df[
            "validation_masked_mse"
        ].idxmin()
    ]
)

print()
print("=" * 88)
print("FOLD 3 MAE TRAINING COMPLETE")
print("=" * 88)

print(
    f"Final TRAIN masked MSE : "
    f"{history_df.iloc[-1]['train_masked_mse']:.8f}"
)

print(
    f"Final VAL masked MSE   : "
    f"{history_df.iloc[-1]['validation_masked_mse']:.8f}"
)

print(
    f"Minimum VAL masked MSE : "
    f"{best_val_row['validation_masked_mse']:.8f}"
)

print(
    f"Epoch of minimum VAL   : "
    f"{int(best_val_row['epoch'])}"
)

print(
    f"Training time          : "
    f"{training_seconds:.2f} s"
)

print()
print(
    f"Checkpoint:\n{checkpoint_path}"
)

print()
print(
    f"History:\n{history_path}"
)

print()
print("=" * 88)
print(
    "STATUS: FOLD 3 MAE TRAINING PASSED"
)
print("=" * 88)
print(
    "The final epoch-300 checkpoint is retained. "
    "Validation loss was diagnostic only and was not used "
    "for early stopping or model selection."
)

In [ ]:
# =============================================================================
# CELL 3B
# DEVELOPMENT-ONLY MAE LEARNING-RATE / EPOCH SELECTION
#
# PURPOSE
# -------
# The initial LR=1e-3 run showed strong TRAIN/VALIDATION divergence.
#
# Fold 3 is the designated DEVELOPMENT fold, so it may be used to select:
#   1. MAE learning rate
#   2. MAE training epoch
#
# IMPORTANT
# ---------
# Selection uses ONLY masked reconstruction loss.
# No clustering metrics, geology, GPI, warm springs, or outer-fold results
# are used for hyperparameter selection.
#
# After this cell:
#   - LR is frozen
#   - epoch count is frozen
#   - outer folds will NOT use validation loss for checkpoint selection
# =============================================================================

import copy
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader


# -----------------------------------------------------------------------------
# 1. Development search space
# -----------------------------------------------------------------------------

DEV_LR_CANDIDATES = [
    1e-4,
    3e-4,
    1e-3,
]

DEV_MAX_EPOCHS = 300

DEV_WEIGHT_DECAY = 1e-4
DEV_BATCH_SIZE = 32

DEV_SEED = 2026
VAL_MASK_SEED = 123456


# -----------------------------------------------------------------------------
# 2. Output directory
# -----------------------------------------------------------------------------

tuning_dir = (
    OUTPUT_ROOT
    / "Fold_3_DEVELOPMENT"
    / "MAE_training_tuning"
)

tuning_dir.mkdir(
    parents=True,
    exist_ok=True
)


# -----------------------------------------------------------------------------
# 3. Helper: deterministic state reset
# -----------------------------------------------------------------------------

def reset_all_seeds(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# -----------------------------------------------------------------------------
# 4. Helper: create identical shuffled TRAIN loader for each experiment
# -----------------------------------------------------------------------------

def make_train_loader(seed):

    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        train_dataset,
        batch_size=DEV_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        generator=generator,
        drop_last=False,
    )


# -----------------------------------------------------------------------------
# 5. Helper: validation with a fixed mask sequence
# -----------------------------------------------------------------------------

@torch.no_grad()
def fixed_validation_loss(
    model,
    loader,
    seed=VAL_MASK_SEED
):

    model.eval()

    cpu_state = torch.get_rng_state()

    cuda_state = None

    if torch.cuda.is_available():
        cuda_state = torch.cuda.get_rng_state_all()

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    loss_sum = 0.0
    n_total = 0

    for batch in loader:

        batch = batch.to(
            DEVICE,
            non_blocking=True
        )

        output = model(batch)

        batch_n = batch.shape[0]

        loss_sum += (
            float(output["loss"].item())
            * batch_n
        )

        n_total += batch_n

    # Restore RNG state
    torch.set_rng_state(cpu_state)

    if (
        torch.cuda.is_available()
        and cuda_state is not None
    ):
        torch.cuda.set_rng_state_all(
            cuda_state
        )

    return loss_sum / n_total


# -----------------------------------------------------------------------------
# 6. Fixed validation loader
# -----------------------------------------------------------------------------

val_loader_tuning = DataLoader(
    val_dataset,
    batch_size=DEV_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)


# -----------------------------------------------------------------------------
# 7. Development sweep
# -----------------------------------------------------------------------------

all_summaries = []

print("=" * 96)
print("FOLD 3 — VANILLA MAE DEVELOPMENT TUNING")
print("=" * 96)

print(f"TRAIN samples       : {len(train_dataset)}")
print(f"VALIDATION samples  : {len(val_dataset)}")
print(f"Learning rates      : {DEV_LR_CANDIDATES}")
print(f"Maximum epochs/run  : {DEV_MAX_EPOCHS}")
print(f"Weight decay        : {DEV_WEIGHT_DECAY}")
print(f"Batch size          : {DEV_BATCH_SIZE}")
print(f"Model parameters    : 207,182")
print()


for lr in DEV_LR_CANDIDATES:

    print("-" * 96)
    print(f"STARTING LR = {lr:.1e}")
    print("-" * 96)

    # -------------------------------------------------------------------------
    # Identical initialization for every LR
    # -------------------------------------------------------------------------

    reset_all_seeds(
        DEV_SEED
    )

    model_lr = VanillaMAE13x13(
        patch_size=PATCH_SIZE,
        n_channels=N_CHANNELS,
        embed_dim=MAE_EMBED_DIM,
        encoder_depth=MAE_ENCODER_DEPTH,
        encoder_heads=MAE_ENCODER_HEADS,
        encoder_mlp_dim=MAE_ENCODER_MLP_DIM,
        decoder_dim=MAE_DECODER_DIM,
        decoder_depth=MAE_DECODER_DEPTH,
        decoder_heads=MAE_DECODER_HEADS,
        decoder_mlp_dim=MAE_DECODER_MLP_DIM,
        n_masked=N_MASKED,
        dropout=MAE_DROPOUT,
    ).to(DEVICE)

    optimizer_lr = torch.optim.AdamW(
        model_lr.parameters(),
        lr=lr,
        weight_decay=DEV_WEIGHT_DECAY
    )

    train_loader_lr = make_train_loader(
        DEV_SEED
    )

    # -------------------------------------------------------------------------
    # Tracking
    # -------------------------------------------------------------------------

    best_val = float("inf")
    best_epoch = None
    best_train = None
    best_state = None

    history_lr = []

    run_start = time.time()

    # -------------------------------------------------------------------------
    # Training
    # -------------------------------------------------------------------------

    for epoch in range(
        1,
        DEV_MAX_EPOCHS + 1
    ):

        model_lr.train()

        train_loss_sum = 0.0
        train_n = 0

        for batch in train_loader_lr:

            batch = batch.to(
                DEVICE,
                non_blocking=True
            )

            optimizer_lr.zero_grad(
                set_to_none=True
            )

            output = model_lr(batch)

            loss = output["loss"]

            if not torch.isfinite(loss):

                raise RuntimeError(
                    f"Non-finite loss: "
                    f"LR={lr}, epoch={epoch}"
                )

            loss.backward()

            optimizer_lr.step()

            batch_n = batch.shape[0]

            train_loss_sum += (
                float(loss.item())
                * batch_n
            )

            train_n += batch_n


        train_loss = (
            train_loss_sum
            / train_n
        )


        # ---------------------------------------------------------------------
        # Fixed validation masking
        # ---------------------------------------------------------------------

        val_loss = fixed_validation_loss(
            model_lr,
            val_loader_tuning,
            seed=VAL_MASK_SEED
        )


        history_lr.append(
            {
                "learning_rate": lr,
                "epoch": epoch,
                "train_masked_mse": train_loss,
                "validation_masked_mse": val_loss,
            }
        )


        # ---------------------------------------------------------------------
        # Development checkpoint selection
        # ---------------------------------------------------------------------

        if val_loss < best_val:

            best_val = val_loss
            best_epoch = epoch
            best_train = train_loss

            best_state = copy.deepcopy(
                {
                    key: value.detach().cpu()
                    for key, value
                    in model_lr.state_dict().items()
                }
            )


        # ---------------------------------------------------------------------
        # Logging
        # ---------------------------------------------------------------------

        if (
            epoch == 1
            or epoch % 25 == 0
            or epoch == DEV_MAX_EPOCHS
        ):

            marker = ""

            if epoch == best_epoch:
                marker = "  *CURRENT BEST*"

            print(
                f"LR={lr:.1e} | "
                f"Epoch {epoch:03d}/{DEV_MAX_EPOCHS} | "
                f"train={train_loss:.8f} | "
                f"val={val_loss:.8f}"
                f"{marker}"
            )


    # -------------------------------------------------------------------------
    # Save this LR history
    # -------------------------------------------------------------------------

    history_lr_df = pd.DataFrame(
        history_lr
    )

    lr_tag = (
        f"{lr:.0e}"
        .replace("-", "m")
    )

    history_lr_path = (
        tuning_dir
        / f"MAE_Fold3_LR_{lr_tag}_history.csv"
    )

    history_lr_df.to_csv(
        history_lr_path,
        index=False
    )


    # -------------------------------------------------------------------------
    # Save best checkpoint for this LR
    # -------------------------------------------------------------------------

    best_checkpoint_path = (
        tuning_dir
        / f"MAE_Fold3_LR_{lr_tag}_BEST.pt"
    )

    torch.save(
        {
            "model_state_dict":
                best_state,

            "learning_rate":
                lr,

            "best_epoch":
                best_epoch,

            "best_train_masked_mse":
                best_train,

            "best_validation_masked_mse":
                best_val,

            "weight_decay":
                DEV_WEIGHT_DECAY,

            "batch_size":
                DEV_BATCH_SIZE,

            "seed":
                DEV_SEED,

            "validation_mask_seed":
                VAL_MASK_SEED,
        },
        best_checkpoint_path
    )


    elapsed = (
        time.time()
        - run_start
    )


    all_summaries.append(
        {
            "learning_rate": lr,
            "best_epoch": best_epoch,
            "best_train_masked_mse":
                best_train,
            "best_validation_masked_mse":
                best_val,
            "run_seconds":
                elapsed,
            "checkpoint":
                str(best_checkpoint_path),
        }
    )


    print()
    print(
        f"LR={lr:.1e} BEST:"
    )

    print(
        f"  epoch     = {best_epoch}"
    )

    print(
        f"  train MSE = {best_train:.8f}"
    )

    print(
        f"  val MSE   = {best_val:.8f}"
    )

    print(
        f"  time      = {elapsed:.2f} s"
    )

    print()


# -----------------------------------------------------------------------------
# 8. Compare development candidates
# -----------------------------------------------------------------------------

summary_df = pd.DataFrame(
    all_summaries
).sort_values(
    "best_validation_masked_mse",
    ascending=True
).reset_index(drop=True)


print("=" * 96)
print("DEVELOPMENT TUNING SUMMARY")
print("=" * 96)

display(
    summary_df[
        [
            "learning_rate",
            "best_epoch",
            "best_train_masked_mse",
            "best_validation_masked_mse",
            "run_seconds",
        ]
    ]
)


# -----------------------------------------------------------------------------
# 9. Select winner ONLY by Fold-3 masked validation reconstruction loss
# -----------------------------------------------------------------------------

winner = summary_df.iloc[0]

SELECTED_MAE_LR = float(
    winner["learning_rate"]
)

SELECTED_MAE_EPOCHS = int(
    winner["best_epoch"]
)

SELECTED_MAE_VAL_MSE = float(
    winner["best_validation_masked_mse"]
)

SELECTED_MAE_CHECKPOINT = Path(
    winner["checkpoint"]
)


# -----------------------------------------------------------------------------
# 10. Save frozen development selection
# -----------------------------------------------------------------------------

selection_csv = (
    tuning_dir
    / "MAE_Fold3_development_selection.csv"
)

summary_df.to_csv(
    selection_csv,
    index=False
)


selection_json = (
    tuning_dir
    / "MAE_Fold3_FROZEN_training_configuration.json"
)

with open(
    selection_json,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "selection_basis":
                "Minimum Fold-3 masked validation reconstruction MSE",

            "learning_rate":
                SELECTED_MAE_LR,

            "epochs":
                SELECTED_MAE_EPOCHS,

            "weight_decay":
                DEV_WEIGHT_DECAY,

            "batch_size":
                DEV_BATCH_SIZE,

            "mask_ratio":
                MASK_RATIO,

            "n_masked":
                N_MASKED,

            "latent_dim":
                MAE_EMBED_DIM,

            "development_fold":
                3,

            "outer_folds":
                [1, 2, 4, 5],

            "best_validation_masked_mse":
                SELECTED_MAE_VAL_MSE,

            "selected_checkpoint":
                str(
                    SELECTED_MAE_CHECKPOINT
                ),

            "rule_for_outer_folds":
                (
                    "Use frozen learning rate and epoch count. "
                    "Do not select checkpoints using outer-fold "
                    "validation loss."
                ),
        },
        f,
        indent=4
    )


# -----------------------------------------------------------------------------
# 11. Boundary check
# -----------------------------------------------------------------------------

selection_at_boundary = (
    SELECTED_MAE_EPOCHS
    == DEV_MAX_EPOCHS
)


print()
print("=" * 96)
print("FROZEN MAE DEVELOPMENT SELECTION")
print("=" * 96)

print(
    f"Selected LR          : "
    f"{SELECTED_MAE_LR:.1e}"
)

print(
    f"Selected epochs      : "
    f"{SELECTED_MAE_EPOCHS}"
)

print(
    f"Best Fold-3 VAL MSE  : "
    f"{SELECTED_MAE_VAL_MSE:.8f}"
)

print(
    f"Selected checkpoint  :\n"
    f"{SELECTED_MAE_CHECKPOINT}"
)

print()
print(
    f"Selection CSV:\n{selection_csv}"
)

print()
print(
    f"Frozen configuration:\n"
    f"{selection_json}"
)


if selection_at_boundary:

    print()
    print(
        "WARNING: selected epoch is the maximum tested epoch. "
        "Training horizon should be extended before freezing."
    )

else:

    print()
    print("=" * 96)
    print(
        "STATUS: MAE DEVELOPMENT TRAINING CONFIGURATION FROZEN"
    )
    print("=" * 96)

    print(
        "Outer folds must use this LR and epoch count exactly. "
        "Outer validation losses must not be used for checkpoint selection."
    )

In [ ]:
# =============================================================================
# CELL 4
# Extract frozen Fold-3 TRAIN and VALIDATION MAE embeddings
#
# IMPORTANT
# ---------
# - Uses the DEVELOPMENT-selected MAE checkpoint only.
# - Uses encode_full(): all 169 spatial tokens are available.
# - No random masking is applied during representation extraction.
# - No latent-space scaling or standardization is applied.
# - No clustering occurs in this cell.
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader


# -----------------------------------------------------------------------------
# 1. Frozen development selection
# -----------------------------------------------------------------------------

FROZEN_MAE_LR = 1e-3
FROZEN_MAE_EPOCHS = 6

FOLD3_SELECTED_CHECKPOINT = (
    OUTPUT_ROOT
    / "Fold_3_DEVELOPMENT"
    / "MAE_training_tuning"
    / "MAE_Fold3_LR_1em03_BEST.pt"
)

assert FOLD3_SELECTED_CHECKPOINT.exists(), (
    f"Selected checkpoint not found:\n"
    f"{FOLD3_SELECTED_CHECKPOINT}"
)


# -----------------------------------------------------------------------------
# 2. Reload selected checkpoint into a clean model instance
# -----------------------------------------------------------------------------

mae_fold3 = VanillaMAE13x13(
    patch_size=PATCH_SIZE,
    n_channels=N_CHANNELS,
    embed_dim=MAE_EMBED_DIM,
    encoder_depth=MAE_ENCODER_DEPTH,
    encoder_heads=MAE_ENCODER_HEADS,
    encoder_mlp_dim=MAE_ENCODER_MLP_DIM,
    decoder_dim=MAE_DECODER_DIM,
    decoder_depth=MAE_DECODER_DEPTH,
    decoder_heads=MAE_DECODER_HEADS,
    decoder_mlp_dim=MAE_DECODER_MLP_DIM,
    n_masked=N_MASKED,
    dropout=MAE_DROPOUT,
).to(DEVICE)


checkpoint = torch.load(
    FOLD3_SELECTED_CHECKPOINT,
    map_location=DEVICE
)

mae_fold3.load_state_dict(
    checkpoint["model_state_dict"]
)

mae_fold3.eval()


# -----------------------------------------------------------------------------
# 3. Verify frozen checkpoint provenance
# -----------------------------------------------------------------------------

print("=" * 92)
print("FOLD 3 — SELECTED MAE CHECKPOINT")
print("=" * 92)

print(
    f"Checkpoint LR          : "
    f"{checkpoint['learning_rate']}"
)

print(
    f"Checkpoint best epoch  : "
    f"{checkpoint['best_epoch']}"
)

print(
    f"Checkpoint TRAIN MSE   : "
    f"{checkpoint['best_train_masked_mse']:.8f}"
)

print(
    f"Checkpoint VAL MSE     : "
    f"{checkpoint['best_validation_masked_mse']:.8f}"
)

assert np.isclose(
    checkpoint["learning_rate"],
    FROZEN_MAE_LR
)

assert (
    int(checkpoint["best_epoch"])
    == FROZEN_MAE_EPOCHS
)


# -----------------------------------------------------------------------------
# 4. Non-shuffled loaders for deterministic embedding extraction
# -----------------------------------------------------------------------------

embedding_batch_size = 64

train_embed_loader = DataLoader(
    train_dataset,
    batch_size=embedding_batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)

val_embed_loader = DataLoader(
    val_dataset,
    batch_size=embedding_batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)


# -----------------------------------------------------------------------------
# 5. Representation extraction helper
# -----------------------------------------------------------------------------

@torch.no_grad()
def extract_full_embeddings(
    model,
    loader
):
    """
    Extract one deterministic 64-D representation per patch
    using the full unmasked 13x13 input.
    """

    model.eval()

    embeddings = []

    for batch in loader:

        batch = batch.to(
            DEVICE,
            non_blocking=True
        )

        latent = model.encode_full(
            batch
        )

        embeddings.append(
            latent.detach().cpu().numpy()
        )

    return np.concatenate(
        embeddings,
        axis=0
    ).astype(np.float32)


# -----------------------------------------------------------------------------
# 6. Extract TRAIN and VALIDATION embeddings
# -----------------------------------------------------------------------------

Z_train = extract_full_embeddings(
    mae_fold3,
    train_embed_loader
)

Z_val = extract_full_embeddings(
    mae_fold3,
    val_embed_loader
)


print()
print("=" * 92)
print("FOLD 3 — RAW MAE LATENT SHAPES")
print("=" * 92)

print(
    "TRAIN embedding shape     :",
    Z_train.shape
)

print(
    "VALIDATION embedding shape:",
    Z_val.shape
)

print(
    "TRAIN dtype               :",
    Z_train.dtype
)

print(
    "VALIDATION dtype          :",
    Z_val.dtype
)


# -----------------------------------------------------------------------------
# 7. Hard numerical checks
# -----------------------------------------------------------------------------

assert Z_train.shape == (
    len(train_dataset),
    MAE_EMBED_DIM
)

assert Z_val.shape == (
    len(val_dataset),
    MAE_EMBED_DIM
)

assert np.isfinite(
    Z_train
).all()

assert np.isfinite(
    Z_val
).all()


# -----------------------------------------------------------------------------
# 8. Representation-collapse and transfer diagnostics
# -----------------------------------------------------------------------------

train_dim_mean = Z_train.mean(
    axis=0
)

val_dim_mean = Z_val.mean(
    axis=0
)

train_dim_std = Z_train.std(
    axis=0,
    ddof=1
)

val_dim_std = Z_val.std(
    axis=0,
    ddof=1
)


# Dimensions that have essentially collapsed
STD_EPS = 1e-6

train_near_zero_std = int(
    np.sum(
        train_dim_std < STD_EPS
    )
)

val_near_zero_std = int(
    np.sum(
        val_dim_std < STD_EPS
    )
)


# -------------------------------------------------------------------------
# Validation-to-training SD transfer ratio
#
# Same conceptual diagnostic previously used for PB-MMAE:
# how much latent dispersion is retained geographically?
# -------------------------------------------------------------------------

valid_sd_dims = (
    train_dim_std > STD_EPS
)

sd_ratio_per_dim = (
    val_dim_std[valid_sd_dims]
    / train_dim_std[valid_sd_dims]
)

mean_sd_ratio = float(
    np.mean(sd_ratio_per_dim)
)

median_sd_ratio = float(
    np.median(sd_ratio_per_dim)
)

sd_ratio_std = float(
    np.std(
        sd_ratio_per_dim,
        ddof=1
    )
)


# -------------------------------------------------------------------------
# Latent vector norms
# -------------------------------------------------------------------------

train_norms = np.linalg.norm(
    Z_train,
    axis=1
)

val_norms = np.linalg.norm(
    Z_val,
    axis=1
)


# -------------------------------------------------------------------------
# Global spread
# -------------------------------------------------------------------------

train_global_std = float(
    Z_train.std(ddof=1)
)

val_global_std = float(
    Z_val.std(ddof=1)
)


audit_summary = pd.DataFrame(
    [
        {
            "Model": "Vanilla_MAE",
            "Fold": 3,
            "Split": "TRAIN",
            "N": Z_train.shape[0],
            "Latent_dim": Z_train.shape[1],
            "Global_mean":
                float(Z_train.mean()),
            "Global_std":
                train_global_std,
            "Median_dim_std":
                float(
                    np.median(
                        train_dim_std
                    )
                ),
            "Mean_vector_norm":
                float(
                    train_norms.mean()
                ),
            "Median_vector_norm":
                float(
                    np.median(
                        train_norms
                    )
                ),
            "Near_zero_SD_dims":
                train_near_zero_std,
        },
        {
            "Model": "Vanilla_MAE",
            "Fold": 3,
            "Split": "VALIDATION",
            "N": Z_val.shape[0],
            "Latent_dim": Z_val.shape[1],
            "Global_mean":
                float(Z_val.mean()),
            "Global_std":
                val_global_std,
            "Median_dim_std":
                float(
                    np.median(
                        val_dim_std
                    )
                ),
            "Mean_vector_norm":
                float(
                    val_norms.mean()
                ),
            "Median_vector_norm":
                float(
                    np.median(
                        val_norms
                    )
                ),
            "Near_zero_SD_dims":
                val_near_zero_std,
        },
    ]
)


transfer_summary = pd.DataFrame(
    [
        {
            "Model":
                "Vanilla_MAE",
            "Fold":
                3,
            "TRAIN_n":
                Z_train.shape[0],
            "VALIDATION_n":
                Z_val.shape[0],
            "Latent_dim":
                MAE_EMBED_DIM,
            "Mean_VAL_to_TRAIN_SD_ratio":
                mean_sd_ratio,
            "Median_VAL_to_TRAIN_SD_ratio":
                median_sd_ratio,
            "SD_of_dimensionwise_ratio":
                sd_ratio_std,
            "TRAIN_near_zero_SD_dims":
                train_near_zero_std,
            "VALIDATION_near_zero_SD_dims":
                val_near_zero_std,
        }
    ]
)


print()
print("=" * 92)
print("LATENT REPRESENTATION AUDIT")
print("=" * 92)

display(audit_summary)

print()
print("=" * 92)
print("GEOGRAPHIC REPRESENTATION-TRANSFER AUDIT")
print("=" * 92)

display(transfer_summary)


# -----------------------------------------------------------------------------
# 9. Save embeddings
# -----------------------------------------------------------------------------

embedding_dir = (
    OUTPUT_ROOT
    / "Fold_3_DEVELOPMENT"
    / "embeddings"
)

embedding_dir.mkdir(
    parents=True,
    exist_ok=True
)


train_embedding_file = (
    embedding_dir
    / "MAE_Fold3_TRAIN_13x13_latent64.npy"
)

val_embedding_file = (
    embedding_dir
    / "MAE_Fold3_VALIDATION_13x13_latent64.npy"
)

np.save(
    train_embedding_file,
    Z_train
)

np.save(
    val_embedding_file,
    Z_val
)


# -----------------------------------------------------------------------------
# 10. Save metadata copies in identical embedding order
# -----------------------------------------------------------------------------

train_meta = pd.read_csv(
    train_meta_file
)

val_meta = pd.read_csv(
    val_meta_file
)

assert len(train_meta) == len(Z_train)
assert len(val_meta) == len(Z_val)


train_meta_out = (
    embedding_dir
    / "MAE_Fold3_TRAIN_13x13_metadata.csv"
)

val_meta_out = (
    embedding_dir
    / "MAE_Fold3_VALIDATION_13x13_metadata.csv"
)

train_meta.to_csv(
    train_meta_out,
    index=False
)

val_meta.to_csv(
    val_meta_out,
    index=False
)


# -----------------------------------------------------------------------------
# 11. Save audit tables
# -----------------------------------------------------------------------------

audit_file = (
    embedding_dir
    / "MAE_Fold3_latent_distribution_audit.csv"
)

transfer_file = (
    embedding_dir
    / "MAE_Fold3_latent_transfer_audit.csv"
)

audit_summary.to_csv(
    audit_file,
    index=False
)

transfer_summary.to_csv(
    transfer_file,
    index=False
)


# -----------------------------------------------------------------------------
# 12. Reload verification
# -----------------------------------------------------------------------------

Z_train_check = np.load(
    train_embedding_file
)

Z_val_check = np.load(
    val_embedding_file
)

assert np.array_equal(
    Z_train,
    Z_train_check
)

assert np.array_equal(
    Z_val,
    Z_val_check
)


# -----------------------------------------------------------------------------
# 13. Final output
# -----------------------------------------------------------------------------

print()
print("=" * 92)
print("SAVED FOLD-3 MAE REPRESENTATIONS")
print("=" * 92)

print(
    f"TRAIN:\n{train_embedding_file}"
)

print()
print(
    f"VALIDATION:\n{val_embedding_file}"
)

print()
print(
    f"Transfer audit:\n{transfer_file}"
)

print()
print("=" * 92)
print(
    "STATUS: FOLD 3 MAE EMBEDDING EXTRACTION PASSED"
)
print("=" * 92)

print(
    "Raw 64-D embeddings were retained without "
    "latent-space standardization."
)

In [ ]:
# =============================================================================
# CELL 5
# Train frozen Vanilla MAE on OUTER FOLDS 1, 2, 4, and 5
# and extract deterministic full-input 64-D embeddings
#
# FROZEN FROM FOLD-3 DEVELOPMENT
# --------------------------------
# LR           : 1e-3
# Epochs       : 6
# Weight decay : 1e-4
# Batch size   : 32
# Masking      : 68 / 169 spatial tokens
# Latent       : 64-D
#
# IMPORTANT
# ---------
# - No outer-fold hyperparameter tuning.
# - No early stopping.
# - No checkpoint selection using outer validation loss.
# - Final epoch-6 checkpoint is retained for every outer fold.
# - Validation loss is reported only as a diagnostic.
# =============================================================================

from pathlib import Path
import random
import time

import numpy as np
import pandas as pd
import torch

from torch.utils.data import (
    Dataset,
    DataLoader
)


# -----------------------------------------------------------------------------
# 1. Frozen configuration
# -----------------------------------------------------------------------------

OUTER_FOLDS = [1, 2, 4, 5]

FROZEN_MAE_LR = 1e-3
FROZEN_MAE_EPOCHS = 6
FROZEN_MAE_WEIGHT_DECAY = 1e-4
FROZEN_MAE_BATCH_SIZE = 32

OUTER_BASE_SEED = 2026

VAL_MASK_SEED = 123456


print("=" * 96)
print("VANILLA MAE — FROZEN OUTER-FOLD TRAINING")
print("=" * 96)

print("Outer folds       :", OUTER_FOLDS)
print("Learning rate     :", FROZEN_MAE_LR)
print("Epochs            :", FROZEN_MAE_EPOCHS)
print("Weight decay      :", FROZEN_MAE_WEIGHT_DECAY)
print("Batch size        :", FROZEN_MAE_BATCH_SIZE)
print("Masked tokens     :", N_MASKED)
print("Latent dimension  :", MAE_EMBED_DIM)
print("Device            :", DEVICE)


# -----------------------------------------------------------------------------
# 2. Dataset helper
# -----------------------------------------------------------------------------

class FrozenPatchDataset(Dataset):

    def __init__(self, npy_path):

        self.x = np.load(
            npy_path,
            mmap_mode="r"
        )

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):

        sample = np.array(
            self.x[idx],
            dtype=np.float32,
            copy=True
        )

        return torch.from_numpy(
            sample
        )


# -----------------------------------------------------------------------------
# 3. Seed helper
# -----------------------------------------------------------------------------

def reset_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# -----------------------------------------------------------------------------
# 4. Diagnostic validation loss
# -----------------------------------------------------------------------------

@torch.no_grad()
def fixed_masked_validation_loss(
    model,
    loader,
    seed=VAL_MASK_SEED
):

    model.eval()

    cpu_state = torch.get_rng_state()

    cuda_state = None

    if torch.cuda.is_available():
        cuda_state = (
            torch.cuda.get_rng_state_all()
        )

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    total_loss = 0.0
    total_n = 0

    for batch in loader:

        batch = batch.to(
            DEVICE,
            non_blocking=True
        )

        output = model(batch)

        n = batch.shape[0]

        total_loss += (
            float(output["loss"].item())
            * n
        )

        total_n += n

    torch.set_rng_state(
        cpu_state
    )

    if (
        torch.cuda.is_available()
        and cuda_state is not None
    ):
        torch.cuda.set_rng_state_all(
            cuda_state
        )

    return total_loss / total_n


# -----------------------------------------------------------------------------
# 5. Full-input deterministic representation extraction
# -----------------------------------------------------------------------------

@torch.no_grad()
def extract_embeddings(
    model,
    loader
):

    model.eval()

    all_z = []

    for batch in loader:

        batch = batch.to(
            DEVICE,
            non_blocking=True
        )

        z = model.encode_full(
            batch
        )

        all_z.append(
            z.detach()
            .cpu()
            .numpy()
        )

    return np.concatenate(
        all_z,
        axis=0
    ).astype(np.float32)


# -----------------------------------------------------------------------------
# 6. Outer-fold loop
# -----------------------------------------------------------------------------

outer_records = []

for fold in OUTER_FOLDS:

    print()
    print("=" * 96)
    print(f"OUTER FOLD {fold}")
    print("=" * 96)

    fold_seed = (
        OUTER_BASE_SEED
        + fold
    )

    reset_seed(
        fold_seed
    )

    # -------------------------------------------------------------------------
    # Paths
    # -------------------------------------------------------------------------

    fold_dir = (
        PATCH_ROOT
        / f"Fold_{fold}"
    )

    train_file = (
        fold_dir
        / f"Fold{fold}_TRAIN_13x13_14ch.npy"
    )

    val_file = (
        fold_dir
        / f"Fold{fold}_VALIDATION_13x13_14ch.npy"
    )

    train_meta_file = (
        fold_dir
        / f"Fold{fold}_TRAIN_13x13_metadata.csv"
    )

    val_meta_file = (
        fold_dir
        / f"Fold{fold}_VALIDATION_13x13_metadata.csv"
    )

    for p in [
        train_file,
        val_file,
        train_meta_file,
        val_meta_file,
    ]:

        assert p.exists(), p


    # -------------------------------------------------------------------------
    # Datasets
    # -------------------------------------------------------------------------

    train_ds = FrozenPatchDataset(
        train_file
    )

    val_ds = FrozenPatchDataset(
        val_file
    )


    # -------------------------------------------------------------------------
    # Deterministic loaders
    # -------------------------------------------------------------------------

    train_generator = (
        torch.Generator()
    )

    train_generator.manual_seed(
        fold_seed
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=FROZEN_MAE_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        generator=train_generator,
        drop_last=False
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=FROZEN_MAE_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        drop_last=False
    )


    # -------------------------------------------------------------------------
    # Fresh independent MAE model
    # -------------------------------------------------------------------------

    reset_seed(
        fold_seed
    )

    model = VanillaMAE13x13(
        patch_size=PATCH_SIZE,
        n_channels=N_CHANNELS,
        embed_dim=MAE_EMBED_DIM,
        encoder_depth=MAE_ENCODER_DEPTH,
        encoder_heads=MAE_ENCODER_HEADS,
        encoder_mlp_dim=MAE_ENCODER_MLP_DIM,
        decoder_dim=MAE_DECODER_DIM,
        decoder_depth=MAE_DECODER_DEPTH,
        decoder_heads=MAE_DECODER_HEADS,
        decoder_mlp_dim=MAE_DECODER_MLP_DIM,
        n_masked=N_MASKED,
        dropout=MAE_DROPOUT,
    ).to(DEVICE)


    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=FROZEN_MAE_LR,
        weight_decay=FROZEN_MAE_WEIGHT_DECAY
    )


    # -------------------------------------------------------------------------
    # Train EXACTLY 6 epochs
    # -------------------------------------------------------------------------

    history = []

    fold_start = time.time()

    for epoch in range(
        1,
        FROZEN_MAE_EPOCHS + 1
    ):

        model.train()

        total_train_loss = 0.0
        total_train_n = 0

        for batch in train_loader:

            batch = batch.to(
                DEVICE,
                non_blocking=True
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            output = model(
                batch
            )

            loss = output["loss"]

            if not torch.isfinite(loss):

                raise RuntimeError(
                    f"Non-finite loss: "
                    f"Fold {fold}, "
                    f"Epoch {epoch}"
                )

            loss.backward()

            optimizer.step()

            n = batch.shape[0]

            total_train_loss += (
                float(loss.item())
                * n
            )

            total_train_n += n


        train_loss = (
            total_train_loss
            / total_train_n
        )


        # Diagnostic only
        val_loss = (
            fixed_masked_validation_loss(
                model,
                val_loader,
                seed=VAL_MASK_SEED
            )
        )


        history.append(
            {
                "Fold":
                    fold,

                "Epoch":
                    epoch,

                "Train_masked_MSE":
                    train_loss,

                "Validation_masked_MSE":
                    val_loss,
            }
        )


        print(
            f"Epoch {epoch:02d}/"
            f"{FROZEN_MAE_EPOCHS} | "
            f"train={train_loss:.8f} | "
            f"val={val_loss:.8f}"
        )


    training_seconds = (
        time.time()
        - fold_start
    )


    # -------------------------------------------------------------------------
    # Output directories
    # -------------------------------------------------------------------------

    fold_output = (
        OUTPUT_ROOT
        / f"Fold_{fold}_OUTER"
    )

    embedding_dir = (
        fold_output
        / "embeddings"
    )

    fold_output.mkdir(
        parents=True,
        exist_ok=True
    )

    embedding_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    # -------------------------------------------------------------------------
    # Save FINAL epoch-6 checkpoint
    # -------------------------------------------------------------------------

    checkpoint_file = (
        fold_output
        / (
            f"MAE_Fold{fold}_"
            f"13x13_final_epoch"
            f"{FROZEN_MAE_EPOCHS}.pt"
        )
    )

    torch.save(
        {
            "model_state_dict":
                model.state_dict(),

            "fold":
                fold,

            "learning_rate":
                FROZEN_MAE_LR,

            "epochs":
                FROZEN_MAE_EPOCHS,

            "weight_decay":
                FROZEN_MAE_WEIGHT_DECAY,

            "batch_size":
                FROZEN_MAE_BATCH_SIZE,

            "seed":
                fold_seed,

            "selection_rule":
                (
                    "Hyperparameters frozen from "
                    "Fold-3 development; final "
                    "epoch retained without "
                    "outer validation selection."
                ),
        },
        checkpoint_file
    )


    # -------------------------------------------------------------------------
    # Save history
    # -------------------------------------------------------------------------

    history_df = pd.DataFrame(
        history
    )

    history_file = (
        fold_output
        / (
            f"MAE_Fold{fold}_"
            f"training_history.csv"
        )
    )

    history_df.to_csv(
        history_file,
        index=False
    )


    # -------------------------------------------------------------------------
    # Non-shuffled embedding loaders
    # -------------------------------------------------------------------------

    train_embed_loader = DataLoader(
        train_ds,
        batch_size=64,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        drop_last=False
    )

    val_embed_loader = DataLoader(
        val_ds,
        batch_size=64,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        drop_last=False
    )


    # -------------------------------------------------------------------------
    # Full-input 64-D embeddings
    # -------------------------------------------------------------------------

    Z_train_outer = extract_embeddings(
        model,
        train_embed_loader
    )

    Z_val_outer = extract_embeddings(
        model,
        val_embed_loader
    )


    assert Z_train_outer.shape == (
        len(train_ds),
        64
    )

    assert Z_val_outer.shape == (
        len(val_ds),
        64
    )

    assert np.isfinite(
        Z_train_outer
    ).all()

    assert np.isfinite(
        Z_val_outer
    ).all()


    # -------------------------------------------------------------------------
    # Save embeddings
    # -------------------------------------------------------------------------

    train_embedding_file = (
        embedding_dir
        / (
            f"MAE_Fold{fold}_"
            f"TRAIN_13x13_latent64.npy"
        )
    )

    val_embedding_file = (
        embedding_dir
        / (
            f"MAE_Fold{fold}_"
            f"VALIDATION_13x13_latent64.npy"
        )
    )

    np.save(
        train_embedding_file,
        Z_train_outer
    )

    np.save(
        val_embedding_file,
        Z_val_outer
    )


    # -------------------------------------------------------------------------
    # Save metadata in identical order
    # -------------------------------------------------------------------------

    train_meta = pd.read_csv(
        train_meta_file
    )

    val_meta = pd.read_csv(
        val_meta_file
    )

    assert len(train_meta) == len(
        Z_train_outer
    )

    assert len(val_meta) == len(
        Z_val_outer
    )

    train_meta.to_csv(
        embedding_dir
        / (
            f"MAE_Fold{fold}_"
            f"TRAIN_13x13_metadata.csv"
        ),
        index=False
    )

    val_meta.to_csv(
        embedding_dir
        / (
            f"MAE_Fold{fold}_"
            f"VALIDATION_13x13_metadata.csv"
        ),
        index=False
    )


    # -------------------------------------------------------------------------
    # Latent transfer audit
    # -------------------------------------------------------------------------

    train_std = Z_train_outer.std(
        axis=0,
        ddof=1
    )

    val_std = Z_val_outer.std(
        axis=0,
        ddof=1
    )

    valid_dims = (
        train_std > 1e-6
    )

    sd_ratio = (
        val_std[valid_dims]
        / train_std[valid_dims]
    )


    record = {

        "Fold":
            fold,

        "Train_n":
            len(train_ds),

        "Validation_n":
            len(val_ds),

        "Final_train_MSE":
            history_df.iloc[-1][
                "Train_masked_MSE"
            ],

        "Final_validation_MSE":
            history_df.iloc[-1][
                "Validation_masked_MSE"
            ],

        "Mean_VAL_to_TRAIN_SD_ratio":
            float(
                np.mean(sd_ratio)
            ),

        "Median_VAL_to_TRAIN_SD_ratio":
            float(
                np.median(sd_ratio)
            ),

        "TRAIN_near_zero_SD_dims":
            int(
                np.sum(
                    train_std < 1e-6
                )
            ),

        "VALIDATION_near_zero_SD_dims":
            int(
                np.sum(
                    val_std < 1e-6
                )
            ),

        "Training_seconds":
            training_seconds,
    }

    outer_records.append(
        record
    )


    print()
    print(
        f"TRAIN embedding      : "
        f"{Z_train_outer.shape}"
    )

    print(
        f"VALIDATION embedding : "
        f"{Z_val_outer.shape}"
    )

    print(
        "Mean VAL/TRAIN SD    : "
        f"{record['Mean_VAL_to_TRAIN_SD_ratio']:.4f}"
    )

    print(
        f"Training time        : "
        f"{training_seconds:.2f} s"
    )


# -----------------------------------------------------------------------------
# 7. Final outer-fold summary
# -----------------------------------------------------------------------------

outer_summary_df = pd.DataFrame(
    outer_records
)

outer_summary_file = (
    OUTPUT_ROOT
    / "MAE_13x13_outer_fold_training_summary.csv"
)

outer_summary_df.to_csv(
    outer_summary_file,
    index=False
)


print()
print("=" * 96)
print("VANILLA MAE — OUTER-FOLD TRAINING SUMMARY")
print("=" * 96)

display(
    outer_summary_df.round(4)
)


print()
print(
    "Mean outer VAL/TRAIN SD ratio:",
    round(
        outer_summary_df[
            "Mean_VAL_to_TRAIN_SD_ratio"
        ].mean(),
        4
    )
)


# -----------------------------------------------------------------------------
# 8. Hard completion checks
# -----------------------------------------------------------------------------

assert set(
    outer_summary_df["Fold"]
) == {1, 2, 4, 5}

assert (
    outer_summary_df[
        "TRAIN_near_zero_SD_dims"
    ] == 0
).all()

assert (
    outer_summary_df[
        "VALIDATION_near_zero_SD_dims"
    ] == 0
).all()


print()
print("=" * 96)
print(
    "STATUS: FROZEN MAE OUTER-FOLD TRAINING PASSED"
)
print("=" * 96)

print(
    "All outer models used the Fold-3-frozen "
    "LR=1e-3 and six-epoch training budget. "
    "No outer validation checkpoint selection was performed."
)